In [1]:
import numpy as np

# --- Step 1: Resampling ---

def length(points):
    """Corresponds to LENGTH(points)"""
    count = 0
    for _ in points:
        count += 1
    return count

def get_time(N, n):
    """Corresponds to TIME(N, n)"""
    old_time = [float(i) for i in range(N)]
    new_time = [j * (N - 1) / (n - 1) for j in range(n)]
    return np.array(old_time), np.array(new_time)

def resample(points, n):
    """Corresponds to RESAMPLE(points, n) using piecewise linear interpolation"""
    N = length(points)
    old_time, new_time = get_time(N, n)
    new_points = []
    t_o = 0
    
    for tn in new_time:
        # Move through old time indices until we find the interval containing tn
        while t_o < N - 1 and old_time[t_o] < tn:
            t_o += 1
            
        if t_o == 0:
            new_points.append(points[0])
            continue
            
        y1, y2 = points[t_o - 1], points[t_o]
        x1, x2 = old_time[t_o - 1], old_time[t_o]
        
        # Calculate slope (m) and intercept (b)
        m = (y2 - y1) / (x2 - x1)
        b = y2 - m * x2
        q = m * tn + b
        new_points.append(q)
        
    return np.array(new_points)

# --- Step 2: Scaling and Normalization ---

def mean_val(points):
    """Corresponds to MEAN(points)"""
    n = length(points)
    return sum(points) / n if n > 0 else 0

def demean(points):
    """Corresponds to DEMEAN(points)"""
    avg = mean_val(points)
    return np.array([p - avg for p in points])

def calculate_std(biosignal_flat):
    """Corresponds to STD(biosignal)"""
    ix = length(biosignal_flat)
    avg = mean_val(biosignal_flat)
    variance_sum = sum((p - avg)**2 for p in biosignal_flat)
    return np.sqrt(variance_sum / ix)

def normalize(biosignal_channels):
    """
    Corresponds to NORMALIZE(biosignal).
    Input: List of channels (arrays)
    """
    # Flatten all channels to calculate global std
    all_points = np.concatenate(biosignal_channels)
    sigma = calculate_std(all_points)
    
    new_biosignal = []
    for channel in biosignal_channels:
        new_channel = [p / sigma for p in channel]
        new_biosignal.append(np.array(new_channel))
    return new_biosignal

# --- Step 3: PCA Operations ---

def pca_process(D, n_pc):
    """
    Corresponds to PCA(D, c, n, nPC).
    D is (channels x timepoints)
    """
    # 1 & 2: Covariance and Eigenvalues
    # Using numpy optimized versions as suggested by the text
    cov_matrix = np.cov(D) 
    eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
    
    # 3 & 4: Sort and select Top nPC components
    idx = eigenvalues.argsort()[::-1]
    U = eigenvectors[:, idx][:, :n_pc]
    
    # 5 & 6: Transform and Flatten
    # D.T is (n x c), U is (c x n_pc) -> result is (n x n_pc)
    transformed = np.matmul(D.T, U)
    flattened = transformed.flatten() # Row-major flattening
    
    return U, flattened

# --- Step 4: Recognition ---

def distance(ai, bi):
    """Corresponds to DISTANCE(Ai, Bi)"""
    return abs(ai - bi)

def path_distance(A, B):
    """Corresponds to PATH-DISTANCE(A, B)"""
    d = 0.0
    for i in range(len(A)):
        d += distance(A[i], B[i])
    return d

def recognize(gesture_G, templates):
    """
    Corresponds to RECOGNIZE(gesture G, templates).
    gesture_G: matrix (c x n)
    templates: list of dicts containing {'U': pcs, 'points': flattened_template}
    """
    best_dist = float('inf')
    matched_template = None
    
    # G_T is the transpose of the gesture matrix
    G_T = gesture_G.T
    
    for T_i in templates:
        U = T_i['U']
        # Transform candidate G using Template i's Principal Components
        points_G = np.matmul(G_T, U)
        points_G_flat = points_G.flatten()
        
        d = path_distance(points_G_flat, T_i['points'])
        
        if d < best_dist:
            best_dist = d
            matched_template = T_i
            
    return matched_template, best_dist

In [2]:
import numpy as np

# 1. Create dummy biosignal data (3 channels, 50 timepoints)
# Representing a simple "wave" gesture
time_steps = 50
channels = 3
t = np.linspace(0, 1, time_steps)
gesture_data = np.array([np.sin(2 * np.pi * t), np.cos(2 * np.pi * t), np.sin(np.pi * t)])

# 2. Pre-process the template
# Resample to 40 points
resampled_channels = [resample(ch, 40) for ch in gesture_data]

# Demean and Normalize
demeaned_channels = [demean(ch) for ch in resampled_channels]
normalized_channels = normalize(demeaned_channels)

# Convert to matrix D (c x n) for PCA
D = np.array(normalized_channels)

# Compute PCA (Extract top 2 Principal Components)
nPC = 2
U, flattened_template = pca_process(D, nPC)

# Store as a template object
template_1 = {
    'label': 'Wave Gesture',
    'U': U,
    'points': flattened_template
}
templates_list = [template_1]

# 3. Simulate a "Candidate" gesture (The same gesture with noise)
noise = np.random.normal(0, 0.1, (channels, time_steps))
candidate_gesture = gesture_data + noise

# Resample and Normalize candidate to match template constraints
candidate_resampled = [resample(ch, 40) for ch in candidate_gesture]
candidate_demeaned = [demean(ch) for ch in candidate_resampled]
candidate_norm = np.array(normalize(candidate_demeaned))

# 4. Run Recognition
match, score = recognize(candidate_norm, templates_list)

print(f"Matched Template: {match['label']}")
print(f"Path Distance Score: {score:.4f}")

Matched Template: Wave Gesture
Path Distance Score: 7.3417
